# A.2 不在訓練資料裡的資訊怎麼使用？


模型訓練時沒看過一家書店後來搬到哪裡，是否只能猜？如果把最新公告放在問題前面，它就有機會直接讀取資料，再回答地址。這與[用例子提示本次規則](https://birdhackor.github.io/tiny-perceptron-vlm/A.1.html)相似：改變的是當前可見前文，不是權重記憶。需要新資訊不一定先重新訓練，但模型仍需讀懂、正確使用你提供的內容。

以架空的小小書店為例，公告說「2027年3月1日搬到青街8號」。本節不主張這是真實事件，只用這份固定資料決定答案。問題是新地址，答案應取青街8號，不是日期，也不是模型可能記得的舊地址。讀資料來回答和閉卷記憶是兩種能力，評估時應分別標明有沒有給公告。

文件很多時，系統可先按問題找相關段落，再把段落放進prompt，讓模型生成答案。這條流程稱Retrieval-Augmented Generation（RAG，檢索增強生成）。檢索是「找資料」，prompt是「本次餵給模型的前文」，生成是「依前文續寫」。下圖從本地文件出發，source ID是來源編號，用於稍後回查；詞匹配是最小找段落的方法，下一節會實作。箭頭表示找出資料、放進前文、回答、核對證據的先後。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

In [ ]:
source = {"id": "公告1", "date": "2027年3月1日", "address": "青街8號"}
context = f"小小書店在{source['date']}搬到{source['address']}。"
prompt = f"[來源{source['id']}] {context}\n問題：書店新地址？請依資料回答並標示來源。"
print(prompt)
print("由資料確定的答案", source["address"])

輸入是人工公告的三個欄位，f字串將它們插入文字。第一次`print(prompt)`會展開兩行：第一行含來源公告1、日期與青街8號，第二行是問題，因為字串內有`\n`換行。第二次print因此是整體第三行，應印「由資料確定的答案 青街8號」。程式沒有生成模型回答，只把模型實際應看到的輸入與真值展開，讓你能檢查「答案所需資料是否已提供」。這也說明若直接用程式取address當模型答覆，會跳過讀懂資料的測試。

正式比較應固定同一模型，分別有公告和無公告生成，保存提示與回答；換公告後，正確答案也應跟著換。若只有有公告時答對，說明系統成功利用輸入，不代表權重永久學到新地址。RAG即使找到文件，模型仍可能用錯欄位、混合舊知識，或答出資料沒支持的細節，後面會分開檢查檢索與生成。

正式RAG組用可逐字核對的小世界測這件事：店名`K017`、地址`C2`、來源`D1`，應答格式是`C2[D1]`。120個店名家族整組分成96／12／12家族，三側分別有768／96／96筆公告練習；干擾店也只從同一側挑選。模型由隨機權重建立，寬64、兩層、兩頭、最多160位置，沒有接續前面SFT或文字預訓練權重。它混合768筆RAG與126筆示例映射訓練資料，每批24筆、學習率0.003、seed 42，真正更新1,000次，累計157,361個有效回答目標。這是按示範回答訓練的新模型。

訓練結束後才替12個未教過的店名重新抽地址與來源，再固定權重生成。有正確公告時答對5/12。將同一店名的公告只改地址，改後答對7/12，但原版與改版都答對只有5/12對；只改來源編號，改後答對5/12，兩版皆對只有2/12對。例如`K017`真的由`C2[D1]`改答`C3[D1]`，`K028`真的由`B9[D9]`改答`B9[D0]`；這些個別成功不足以保證每份新公告都跟得上。改動只作用於當次輸入文件與期待答案，沒有修改保存的資料切分或再次訓練。

在已啟用專案環境的根目錄可完整重跑這份配方；CPU時間與正式L4時間不同，有自己的CUDA環境才將裝置改為`cuda`：

```bash
python scripts/course_experiments/run.py --experiment rag --device cpu
```

`outputs/course-experiments/course-v1/rag/`保存`dataset.json`、`icl-dataset.json`、`retrieval-corpus.json`、`generations.json`、最終`model.pt`及四份中間checkpoint。改設定須另留結果；完整配方與失敗生成見[正式報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/rag.json)。

練習只把source的address改成紅街9號，日期與問題保持不變。先預測prompt與標準答案同步改地址，再執行核對；如果以後用模型測試，回答也應跟隨公告，而不能繼續說青街8號。這個練習只改變本次資料，不訓練任何權重。
